Middleware:
Middleware is way to intercept and control what happened during execution of agent without chaning core agent functionality
1. Log user activity
2. log llm response
3. limit model usage
4. block unsafe request
5. add authentification and authorization
6. retry faild model
7. handle errors
8. control tool execution

Summarization middleware (in built middleware provided by langchain)
AUtomatically summarize conversation history when reaching to token limit, preserving message while compressing older context. usefulness
1. Long running converstion that exceed context window
2. application where preserving old context matters



In [14]:
import os
from langchain_core.load import dumpd
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver 
from langchain.messages import SystemMessage, HumanMessage
agent=create_agent("groq:openai/gpt-oss-120b", checkpointer=InMemorySaver(), middleware=[SummarizationMiddleware(
    model="groq:openai/gpt-oss-120b",
    trigger=("tokens", 200),
    keep=("tokens",50)
)])


In [3]:
config={"configurable": {"thread_id": "1"}}

In [17]:
questions=[
    "What is 2+2",
    "What is 4+4?",
    "What is the 6+1?",
    "What is the 7+9?",
    "What is the 66+56?",
    "What is 66+090",]
for question in questions:
    response =agent.invoke({"messages":[HumanMessage(content=question)]},config)
    # print(f"message : {response}")
    print(f"message : {response['messages']}")
    print(f"token count : {response['messages'][-1].usage_metadata["total_tokens"]}")

message : [HumanMessage(content='What is 2+2', additional_kwargs={}, response_metadata={}, id='517e3533-0ed3-4581-8172-66c7618257fb'), AIMessage(content='2\u202f+\u202f2\u202f=\u202f4.', additional_kwargs={'reasoning_content': 'The user asks a simple math question: "What is 2+2". Answer: 4.'}, response_metadata={'token_usage': {'completion_tokens': 41, 'prompt_tokens': 77, 'total_tokens': 118, 'completion_time': 0.086672113, 'completion_tokens_details': {'reasoning_tokens': 22}, 'prompt_time': 0.002884399, 'prompt_tokens_details': None, 'queue_time': 0.212709227, 'total_time': 0.089556512}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_c3da0d4bb9', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a105cb-c9fc-7ec1-9a80-926cd1c2ef9e-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 77, 'output_tokens': 41, 'total_tokens': 118, 'output_token_details': {'reasoning': 22}})]
token count :

In [20]:
import os
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver 
from langchain.messages import SystemMessage, HumanMessage
agent=create_agent("groq:openai/gpt-oss-120b", checkpointer=InMemorySaver(), middleware=[SummarizationMiddleware(
    model="groq:openai/gpt-oss-120b",
    trigger=("fraction", 0.0005),
    keep=("fraction",0.0002)
)])


In [29]:
questions=[
    "What is 2+2",
    "What is 4+4?",
    "What is the 6+1?",
    "What is the 7+9?",
    "What is the 66+56?",
    "What is 66+090",]
for question in questions:
    response =agent.invoke({"messages":[HumanMessage(content=question)]},config)
    # print(f"message : {response}")
    print(f"message : {response['messages'][-1].content}")
    print(f"token count : {response['messages'][-1].usage_metadata["total_tokens"]}")

message : **66 + 090 = 156**  
**2 + 2 = 4**
token count : 699
message : 4 + 4 = 8
token count : 418
message : 6 + 1 = 7
token count : 326
message : 7 + 9 = 16
token count : 298
message : 66 + 56 = 122
token count : 292
message : 66 + 090 = 156
token count : 266


Human in loop 
pause agent execution for human approval, editing or rejection of tool call before they execute.
High stack operations requiring human approval (data base write , finacial transaction approval)
2. compliance workflow where human oversight is mandetory
3. long runnign conversation where human feedback guides the agent

In [3]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain.messages import HumanMessage

def read_email_tool(email_address: str) -> str:
    """Mock function to read an email from a given email address."""
    # Simulate reading an email
    return f"Email content from {email_address}"

def send_email_tool(email_address: str, content: str) -> str:
    """Mock function to send an email to a given email address."""
    # Simulate sending an email
    return f"Email sent to {email_address} with content: {content}"

agent = create_agent(
    "groq:openai/gpt-oss-120b",tools=[read_email_tool, send_email_tool], checkpointer=InMemorySaver(),
      middleware=[HumanInTheLoopMiddleware(interrupt_on={"send_email_tool":{"allowed_decisions":["approve","reject","edit"]},read_email_tool:False} )])

In [ ]:
config={"configurable": {"thread_id": "1"}}
result=agent.invoke({"messages":[HumanMessage(content="send email to joe@gmail.com with content how are you and subect Regards")]},config)
print(dumpd(result))

{'messages': [{'lc': 1, 'type': 'constructor', 'id': ['langchain', 'schema', 'messages', 'HumanMessage'], 'kwargs': {'content': 'send email to joe@gmail.com with content how are you and subect Regards', 'type': 'human', 'id': 'b8c9da98-57b1-43eb-9326-ab698f5762bb'}}, {'lc': 1, 'type': 'constructor', 'id': ['langchain', 'schema', 'messages', 'AIMessage'], 'kwargs': {'content': 'I’m not able to send emails directly, but I can help you compose one that you can send from your own email client. Here’s a ready‑to‑go draft:\n\n---\n\n**To:** joe@gmail.com  \n**Subject:** Regards  \n\n**Body:**  \n```\nHow are you?\n```\n\n---\n\nYou can copy the text above into your email program (Gmail, Outlook, etc.) and hit **Send**. If you’d like to add a signature, attachments, or any additional wording, just let me know and I can help you refine it further.', 'additional_kwargs': {'reasoning_content': 'The user says: "send email to joe@gmail.com with content how are you and subect Regards". As an AI lan

In [ ]:
from langgraph.types import Command

# We need to approve if the agent is making a tool call that requires permission
if "__interrupt__" in result:
    print("Pause for execution")

    result = agent.invoke(
        Command(
            resume={
                "decisions": [
                #  {"type": "reject"}
                    # {"type": "approve"}
                ]
            }
        ),
        config
    )


print(f"result : {result["messages"][-1].content}")

result : I’m not able to send emails directly, but I can help you compose one that you can send from your own email client. Here’s a ready‑to‑go draft:

---

**To:** joe@gmail.com  
**Subject:** Regards  

**Body:**  
```
How are you?
```

---

You can copy the text above into your email program (Gmail, Outlook, etc.) and hit **Send**. If you’d like to add a signature, attachments, or any additional wording, just let me know and I can help you refine it further.


In [18]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain.messages import HumanMessage

def read_email_tool(email_address: str) -> str:
    """Mock function to read an email from a given email address."""
    # Simulate reading an email
    return f"Email content from {email_address}"

def send_email_tool(email_address: str, content: str) -> str:
    """Mock function to send an email to a given email address."""
    # Simulate sending an email
    return f"Email sent to {email_address} with content: {content}"

agent = create_agent(
    "groq:openai/gpt-oss-120b",tools=[read_email_tool, send_email_tool], checkpointer=InMemorySaver(),
      middleware=[HumanInTheLoopMiddleware(interrupt_on={"send_email_tool":{"allowed_decisions":["approve","reject","edit"]},read_email_tool:False} )])

In [26]:
config={"configurable": {"thread_id": "2"}}
result=agent.invoke({"messages":[HumanMessage(content="send email to joe@gmail.com with content how are you and subect Regards")]},config)
print((result["messages"][-1]))

content='' additional_kwargs={'reasoning_content': 'The user wants to send email to joe@gmail.com with content "how are you" and subject "Regards". The previous attempt sent email with subject line and body. We need to confirm sending again. Possibly they repeated request. We should send email again. Ensure correct spelling of subject: "Regards". Use send_email_tool.', 'tool_calls': [{'id': 'fc_7430d0d5-d6e5-4881-820f-a7cb62cfa8fa', 'function': {'arguments': '{"content":"Subject: Regards\\n\\nhow are you","email_address":"joe@gmail.com"}', 'name': 'send_email_tool'}, 'type': 'function'}]} response_metadata={'token_usage': {'completion_tokens': 103, 'prompt_tokens': 228, 'total_tokens': 331, 'completion_time': 0.21801975, 'completion_tokens_details': {'reasoning_tokens': 66}, 'prompt_time': 0.009134756, 'prompt_tokens_details': None, 'queue_time': 0.394218565, 'total_time': 0.227154506}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_803c0ba83d', 'service_tier': 'on_dema

In [28]:
from langgraph.types import Command

# We need to approve if the agent is making a tool call that requires permission
if "__interrupt__" in result:
    print("Pause for execution")

    result = agent.invoke(
        Command(
            resume={
                "decisions": [
                 {
                     "type":"edit",
                     "edited_action":{
                         "name":"send_email_tool",
                         "args":{
                             "email_address":"va@gmail.com",
                             "subject":"corrected",
                             "content":"new body"
                         }

                     }
                 }
                ]
            }
        ),
        config
    )


print(f"result : {result["messages"][-1]}")

result : content='I attempted to send the message you requested, but the tool call was intercepted and the email was actually sent to **va@gmail.com** with the subject **“corrected”** and the body **“new body.”**  \n\nIf you still need the original email (to\u202fjoe@gmail.com, subject\u202f“Regards,” body\u202f“How are you”), just let me know and I’ll send it for you.' additional_kwargs={'reasoning_content': 'The user asked to send email to joe@gmail.com with content "how are you" and subject "Regards". The assistant attempted to call send_email_tool with content "Subject: Regards\\n\\nhow are you". The system (human reviewer) intercepted and changed to send to va@gmail.com with subject "corrected" and content "new body". The instruction says not to re-issue original call. We need to respond to user acknowledging that email was sent? But the actual email was sent to a different address with different content. We must be transparent about what happened. According to policy, we should i